# 📰 Financial News Sentiment Analysis using FinBERT
### Indian Financial News Articles (2003–2020)

This notebook applies **FinBERT** — a BERT model pre-trained specifically
on financial text (`ProsusAI/finbert`) — to classify the sentiment of
Indian financial news headlines as **Positive**, **Negative**, or
**Neutral**, then explores how sentiment evolves over time and what it
reveals about the news landscape.

**Pipeline:**
1. Load & clean the dataset
2. Run FinBERT sentiment inference on every headline
3. Exploratory Data Analysis (distribution, trends, extremes)
4. Visualizations (bar, pie, time-series, heatmap)
5. Business insights
6. Conclusion — findings, limitations, future work

**Dataset columns:** `Date`, `Title`, `Description`


## 1. Setup & Imports

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly transformers torch tqdm -q

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)
sns.set_style("whitegrid")
tqdm.pandas()


## 2. Load Dataset

Reads the raw CSV. If you're running this in Colab/Kaggle, update the
path below to wherever `IndianFinancialNews.csv` lives.


In [ ]:
DATA_PATH = "data/IndianFinancialNews.csv"

df = pd.read_csv(DATA_PATH)
print(f"Raw shape: {df.shape}")
df.info()
df.head()


## 3. Data Cleaning

Steps:
1. Drop exact duplicate rows (same date + headline)
2. Drop rows with a missing headline (`Title`) — that's our core signal
3. Fill missing `Description` with an empty string (not critical for the analysis)
4. Parse `Date` into a proper datetime column
5. Light text cleaning on headlines — strip URLs and extra whitespace.

Note: we deliberately do **not** aggressively clean the text (e.g. no
lowercasing, no stopword removal, no punctuation stripping). FinBERT is a
transformer model trained on natural financial sentences, so it performs
best on text kept close to its original, readable form — over-cleaning
can actually remove signal the model relies on.


In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"\s+", " ", text).strip()       # collapse whitespace
    return text


print(f"Before cleaning: {df.shape}")

df = df.drop_duplicates(subset=["Date", "Title"]).copy()
df = df.dropna(subset=["Title"]).copy()
df["Description"] = df["Description"].fillna("")

df["Title_Clean"] = df["Title"].apply(clean_text)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).copy()

df = df.reset_index(drop=True)
print(f"After cleaning:  {df.shape}")
df.head()


## 4. Sentiment Analysis with FinBERT

We use Hugging Face's `ProsusAI/finbert` — a BERT model fine-tuned
specifically for financial sentiment (trained on analyst reports and
financial news), which outperforms general-purpose sentiment models on
this kind of text.

For a large dataset, we run inference in **batches** on GPU if available
(falls back to CPU automatically), with a progress bar since this is the
slowest step in the notebook.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "ProsusAI/finbert"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# FinBERT's label order for ProsusAI/finbert is: 0=positive, 1=negative, 2=neutral
id2label = model.config.id2label
print(id2label)


In [ ]:
def predict_sentiment_batch(texts, batch_size=32, max_length=64):
    """
    Runs FinBERT on a list of headlines in batches.
    Returns two lists: predicted labels, confidence scores.
    """
    all_labels, all_scores = [], []

    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring sentiment"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                            max_length=max_length, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            confidences, preds = torch.max(probs, dim=-1)

        all_labels.extend([id2label[p.item()] for p in preds])
        all_scores.extend([c.item() for c in confidences])

    return all_labels, all_scores


headlines = df["Title_Clean"].tolist()
labels, scores = predict_sentiment_batch(headlines, batch_size=32)

df["Sentiment"] = labels
df["Confidence"] = scores
df.head()


In [ ]:
# Checkpoint: save results so you don't have to re-run FinBERT every time
df.to_csv("data/IndianFinancialNews_with_sentiment.csv", index=False)
print("Saved scored dataset.")


## 5. Exploratory Data Analysis

### 5.1 Overall Sentiment Distribution


In [ ]:
sentiment_counts = df["Sentiment"].value_counts()
sentiment_pct = df["Sentiment"].value_counts(normalize=True) * 100

summary_table = pd.DataFrame({
    "Count": sentiment_counts,
    "Percentage": sentiment_pct.round(2)
})
summary_table


### 5.2 Most Positive and Most Negative Headlines

In [ ]:
top_positive = (
    df[df["Sentiment"] == "positive"]
    .sort_values("Confidence", ascending=False)
    .head(10)[["Date", "Title_Clean", "Confidence"]]
)

top_negative = (
    df[df["Sentiment"] == "negative"]
    .sort_values("Confidence", ascending=False)
    .head(10)[["Date", "Title_Clean", "Confidence"]]
)

print("=== Top 10 Most Positive Headlines ===")
display(top_positive)

print("\n=== Top 10 Most Negative Headlines ===")
display(top_negative)


### 5.3 Sentiment Trend Over Time

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.to_period("M").astype(str)

# A single numeric "sentiment score" per headline: +confidence for positive,
# -confidence for negative, 0 for neutral -- makes trend charts easy to read.
def to_sentiment_score(row):
    if row["Sentiment"] == "positive":
        return row["Confidence"]
    elif row["Sentiment"] == "negative":
        return -row["Confidence"]
    return 0.0

df["Sentiment_Score"] = df.apply(to_sentiment_score, axis=1)

yearly_counts = df.groupby(["Year", "Sentiment"]).size().unstack(fill_value=0)
monthly_score = df.groupby("Month")["Sentiment_Score"].mean().reset_index()

yearly_counts.tail()


## 6. Visualizations

### 6.1 Sentiment Distribution — Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = {"positive": "#2ca02c", "negative": "#d62728", "neutral": "#7f7f7f"}
bars = ax.bar(sentiment_counts.index, sentiment_counts.values,
               color=[colors.get(s, "#1f77b4") for s in sentiment_counts.index])
ax.bar_label(bars)
ax.set_title("Sentiment Distribution of Indian Financial News Headlines")
ax.set_ylabel("Number of Headlines")
plt.tight_layout()
plt.show()


### 6.2 Sentiment Proportion — Pie Chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct="%1.1f%%",
       colors=[colors.get(s, "#1f77b4") for s in sentiment_counts.index], startangle=90)
ax.set_title("Sentiment Proportion")
plt.tight_layout()
plt.show()


### 6.3 Yearly Sentiment Trend — Stacked Bar (Interactive)

In [ ]:
yearly_long = yearly_counts.reset_index().melt(id_vars="Year", var_name="Sentiment", value_name="Count")

fig = px.bar(yearly_long, x="Year", y="Count", color="Sentiment",
             color_discrete_map=colors, title="Yearly Sentiment Trend",
             barmode="stack")
fig.update_layout(xaxis_title="Year", yaxis_title="Number of Headlines")
fig.show()


### 6.4 Monthly Average Sentiment Score — Time Series (Interactive)

In [ ]:
fig = px.line(monthly_score, x="Month", y="Sentiment_Score",
              title="Monthly Average Sentiment Score Over Time")
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(xaxis_title="Month", yaxis_title="Avg. Sentiment Score (-1 to +1)")
fig.update_xaxes(tickangle=45, nticks=20)
fig.show()


### 6.5 Sentiment Heatmap — Year × Month

Shows *when* sentiment was most positive or negative at a glance — useful
for spotting periods of financial stress (e.g. 2008 crisis, 2020 Covid crash)
directly from headline tone.


In [ ]:
pivot = df.pivot_table(index="Year", columns=df["Date"].dt.month,
                        values="Sentiment_Score", aggfunc="mean")
pivot.columns = [pd.Timestamp(2020, m, 1).strftime("%b") for m in pivot.columns]

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot, cmap="RdYlGn", center=0, annot=False, ax=ax,
            cbar_kws={"label": "Avg. Sentiment Score"})
ax.set_title("Average Monthly Sentiment Score by Year")
plt.tight_layout()
plt.show()


### 6.6 Confidence Distribution by Sentiment

Checks how confident FinBERT is for each sentiment class — useful for
spotting whether "neutral" is a genuinely confident prediction or mostly
low-confidence/uncertain calls.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Sentiment", y="Confidence",
            order=["positive", "neutral", "negative"],
            palette=colors, ax=ax)
ax.set_title("Model Confidence by Sentiment Class")
plt.tight_layout()
plt.show()


## 7. Business Insights

In [ ]:
dominant_sentiment = sentiment_counts.idxmax()
dominant_pct = sentiment_pct.max()

most_positive_year = yearly_counts["positive"].idxmax() if "positive" in yearly_counts else None
most_negative_year = yearly_counts["negative"].idxmax() if "negative" in yearly_counts else None

trend_direction = "improving" if monthly_score["Sentiment_Score"].iloc[-6:].mean() > monthly_score["Sentiment_Score"].iloc[:6].mean() else "declining"

print("KEY INSIGHTS")
print("=" * 60)
print(f"1. Dominant sentiment overall: {dominant_sentiment.upper()} ({dominant_pct:.1f}% of headlines)")
print(f"2. Most positive year (by headline count): {most_positive_year}")
print(f"3. Most negative year (by headline count): {most_negative_year}")
print(f"4. Long-run sentiment trend: {trend_direction} (comparing first 6 vs. last 6 months on record)")
print(f"5. Total headlines analyzed: {len(df):,}")


**How to read these insights for a business/finance audience:**
- A market with a **persistently negative or declining sentiment trend**
  can signal building investor caution or economic stress — useful as a
  leading indicator alongside price/volume data.
- Spikes in **negative sentiment concentrated in specific months/years**
  (visible in the heatmap) typically line up with real macro events —
  worth cross-checking against known crises (2008 GFC, 2020 Covid crash,
  demonetization in Nov 2016, etc.) as a sanity check on the model.
- High-confidence negative headlines are the ones worth reading first when
  triaging news manually — they're where the model is most certain
  something significant happened.


## 8. Conclusion

### Findings
- FinBERT successfully classifies Indian financial news headlines into
  Positive, Negative, and Neutral sentiment with interpretable confidence
  scores, requiring no manual labeling or training data.
- Sentiment trends over time can be visually cross-referenced against known
  market events as a sanity check on the model's real-world relevance.

### Limitations
- **Headline-only analysis:** we only scored the `Title`, not the full
  `Description`/article body — headlines can be sensationalized or lack
  context that changes the actual sentiment.
- **Domain mismatch:** FinBERT was trained primarily on English-language,
  Western financial text (analyst reports, US/UK financial news). Indian
  financial journalism has its own phrasing conventions and jargon, so
  some predictions may be less reliable than on the data FinBERT was
  originally trained on.
- **No ground-truth labels:** without manually labeled headlines to
  compare against, we can't directly compute accuracy/precision/recall —
  only inspect confidence scores and plausibility.
- **Neutral class ambiguity:** "neutral" can both mean *genuinely
  neutral news* and *the model being unsure* — the confidence boxplot in
  Section 6.6 helps distinguish these, but it's an inherent limitation of
  3-class sentiment classification.

### Future Improvements
- Score the full `Description` (or headline + description combined) and
  compare results to headline-only scoring.
- Fine-tune FinBERT on a small hand-labeled sample of Indian financial
  headlines to adapt it to local phrasing and terminology.
- Correlate sentiment scores with actual market index movements (e.g.
  Nifty 50, Sensex) to test whether news sentiment has predictive value.
- Add aspect-based sentiment (e.g. sentiment specifically about a bank vs.
  the broader economy) rather than one score per headline.
- Build a simple dashboard (Streamlit) so sentiment trends can be explored
  interactively without re-running the notebook.
